In [0]:
# 410-*-.json finden und das erste Paar drucken.

import json
from pathlib import Path
name = "data/sample/410/410_DE_hour_1787522400000.json"
candidates = [Path.cwd() / name, Path.cwd().parent / name]
path = next(p for p in candidates if p.exists())
print(path)
with open(path) as f:
    data = json.load(f)
print(list(data.keys()))
print("hours", len(data["series"]))
print("first row", data["series"][0])

In [0]:
# jedes Paar wird eine Zeile und landet in bronze.smard_raw. Das ist Ingest.
# Zeile --> (filter_id, metric=load, event_time_ms, value_mwh) 
from pyspark.sql import functions as F

rows = [(410, "load", int(ts), val) for ts, val in data["series"]]
df = spark.createDataFrame(rows, ["filter_id", "metric", "event_time_ms", "value_mwh"])
df = df.withColumn("source_file", F.lit(str(path)))
df.show(5)

df.write.mode("overwrite").saveAsTable("bronze.smard_raw")
print("rows", df.count())

In [0]:
# Berliner Uhrzeit, Lücken markieren, Duplikate weg, silver.electricity_hourly. Das ist Putzen.

silver = (
    df.withColumn(
        "event_time",
        F.from_utc_timestamp(F.to_timestamp(F.col("event_time_ms") / 1000), "Europe/Berlin"),
    )
    .withColumn("is_missing_value", F.col("value_mwh").isNull())
    .dropDuplicates(["metric", "event_time"])
)

silver.show(5)
silver.write.mode("overwrite").saveAsTable("silver.electricity_hourly")
print("rows", silver.count())

In [0]:
# Stunden zu Tagen, min/schnitt/max, gold.daily_load. Das ist die Report-Tabelle.

gold = (
    silver.groupBy(F.to_date("event_time").alias("day"))
    .agg(
        F.count("*").alias("hours"),
        F.sum(F.when(F.col("is_missing_value"), 1).otherwise(0)).alias("missing_hours"),
        F.avg("value_mwh").alias("avg_load_mwh"),
        F.min("value_mwh").alias("min_load_mwh"),
        F.max("value_mwh").alias("max_load_mwh"),
    )
)

gold.show()
gold.write.mode("overwrite").saveAsTable("gold.daily_load")
print("days", gold.count())

In [0]:


name = "data/sample/4067/4067_DE_hour_1787522400000.json"
candidates = [Path.cwd() / name, Path.cwd().parent / name]
wind_path = next(p for p in candidates if p.exists())

with open(wind_path) as f:
    wind = json.load(f)

rows = [(4067, "wind_onshore", int(ts), val) for ts, val in wind["series"]]
wind_df = spark.createDataFrame(rows, ["filter_id", "metric", "event_time_ms", "value_mwh"])
wind_df = wind_df.withColumn("source_file", F.lit(str(wind_path)))
wind_df.show(5)

wind_df.write.mode("append").saveAsTable("bronze.smard_raw")
print("bronze rows now", spark.table("bronze.smard_raw").count())

In [0]:

bronze = spark.table("bronze.smard_raw")

silver = (
    bronze.withColumn(
        "event_time",
        F.from_utc_timestamp(F.to_timestamp(F.col("event_time_ms") / 1000), "Europe/Berlin"),
    )
    .withColumn("is_missing_value", F.col("value_mwh").isNull())
    .dropDuplicates(["metric", "event_time"])
)

silver.groupBy("metric").count().show()
silver.write.mode("overwrite").saveAsTable("silver.electricity_hourly")
print("silver rows", silver.count())

In [0]:

hourly = (
    spark.table("silver.electricity_hourly")
    .groupBy("event_time")
    .pivot("metric", ["load", "wind_onshore"])
    .agg(F.first("value_mwh"))
)

gold = (
    hourly.withColumn("wind_share", F.col("wind_onshore") / F.col("load"))
    .groupBy(F.to_date("event_time").alias("day"))
    .agg(
        F.avg("load").alias("avg_load_mwh"),
        F.avg("wind_onshore").alias("avg_wind_mwh"),
        F.avg("wind_share").alias("avg_wind_share"),
        F.max("wind_share").alias("max_wind_share"),
    )
)

gold.show()
gold.write.mode("overwrite").saveAsTable("gold.daily_energy_mix")

In [0]:
hourly = (
    spark.table("silver.electricity_hourly")
    .groupBy("event_time")
    .pivot("metric", ["load", "wind_onshore"])
    .agg(F.first("value_mwh"))
    .filter(F.col("load").isNotNull())
)

gold = (
    hourly.withColumn("wind_share", F.col("wind_onshore") / F.col("load"))
    .groupBy(F.to_date("event_time").alias("day"))
    .agg(
        F.count("*").alias("hours"),
        F.avg("load").alias("avg_load_mwh"),
        F.avg("wind_onshore").alias("avg_wind_mwh"),
        F.avg("wind_share").alias("avg_wind_share"),
        F.max("wind_share").alias("max_wind_share"),
    )
)

gold.show()
gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.daily_energy_mix")


## "Load" Dateien sind im Volume. (410)

In [0]:
# Die Datein sind im Volume (broze/landing/*.json)
vol = "/Volumes/workspace/bronze/landing/" # den Pfad 
files = dbutils.fs.ls(vol)
for f in files:
    print(f.name)
print("n", len(files)) # noch nicht bronze überschreiben

## Alle 13. "Load" Dateien nach "Bronze Layer" schreiben

To Do -
1. Suche Volume Pfad
2. Daten Vorbreiten
3. in die Tabellen Laden

In [0]:
# overwrite lochst das alte Bronze

import json
from pathlib import Path

folder = Path("/Volumes/workspace/bronze/landing") # volume Pfad, in dem die 13 Datein liegen

rows = [] 

# for loop öffnet jede .json, nimmt "series" und macht aus jedem Paar eine Zeile: (410, load, Zeit, Wert, Dateiname)
for path in sorted(folder.glob("*.json")):
    data = json.loads(path.read_text())
    for ts, val in data["series"]:
        rows.append((410, "load", int(ts), val, str(path)))

# "rows" ist erst eine Python List
df = spark.createDataFrame( # createDataFrame macht daraus(from this. --> row) eine Spark-Tabelle mit fünf Spalten
    rows,
    ["filter_id", "metric", "event_time_ms", "value_mwh", "source_file"],
)

print("rows", df.count())

df.groupBy("source_file").count().show(20, truncate=False) # groupby.count() ist nur Kontrolle(check), 13 dateien jede mit etwa(with about) 168 Stunden

# overwrite mode ersetzt(replace) die alte Bronze-Tabelle komplett; Sample ist danach weg.
# overwriteSchema erlaubt(tells), dass Spark die Tabellenform neu schreibt, falls(if) die Spalten nicht 1:1 zur(to the) alten Tabelle passen(match).
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.smard_raw") 


## Silver Layer aus dieser Bronze-Tabelle. (410)

To Do -
1. Lesen Daten auf bronze.smard_raw
2. Daten Vorbereiten - entferne Duplikate, markiere NULL, standardisiere die Einträge, usw
3. Bereinigte Daten in silver_electricity_hourly laden 

In [0]:
# Bronze hat jetzt etwa 90 Tage Last.


from pyspark.sql import functions as F

# die gespeicherten Daten lesen, nicht noch einmal die JSON Datein
bronze = spark.table("bronze.smard_raw")

# vor millisekunde nach Unix-Sekeunde stellt auf Berlin um.
silver = (
    bronze.withColumn(
        "event_time",
        F.from_utc_timestamp(F.to_timestamp(F.col("event_time_ms") / 1000), "Europe/Berlin"),
    )
    .withColumn("is_missing_value", F.col("value_mwh").isNull())
    .dropDuplicates(["metric", "event_time"]) # entferne Duplikate, falls eine Woche zweimal geladen wurde.
)

silver.groupBy("metric").count().show() 
print("min", silver.agg(F.min("event_time")).first()[0])
print("max", silver.agg(F.max("event_time")).first()[0])

# schreibt Datein
silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.electricity_hourly")

## Gold: macht aus sauberen Daten eine Tages-Tabelle für Reports (410)


In [0]:
from pyspark.sql import functions as F

gold = (
    spark.table("silver.electricity_hourly") # liest Silver, nicht Bronze und nicht JSON
    .filter(~F.col("is_missing_value")) # putzen --> wirft Stunden ohne Zahl weg 
    .groupBy(F.to_date("event_time").alias("day"))
    .agg( # packt(puts) alle Stunden dessablen(of the same) Kalendertags in eine Zeile(row).
         
        # sind die Report Zahlen
        F.count("*").alias("hours"),
        F.avg("value_mwh").alias("avg_load_mwh"),
        F.min("value_mwh").alias("min_load_mwh"),
        F.max("value_mwh").alias("max_load_mwh"),
    )
)

print("days", gold.count())
gold.orderBy("day").show(10) # überprüfen - ersten zehn Tage 

# schreibt Datein
gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.daily_load")

## Wind ist im Volume. (Manueller Upload) --> Wind Daten an Bronze anhängen (4067)

In [0]:
import json
from pathlib import Path

folder = Path("/Volumes/workspace/bronze/landing")
rows = []
for path in sorted(folder.glob("4067_*.json")): # liest nur Wind-Dateien aus Volume, nicht aus 410 Folder
    data = json.loads(path.read_text())

    # jades(each) series-paar wird eine Zeile(row) mit filter_id=4067 und metric=wind_onshore
    for ts, val in data["series"]:
        rows.append((4067, "wind_onshore", int(ts), val, str(path)))

wind_df = spark.createDataFrame( # createDataFrame macht daraus(from this --> row) eine Spark-Tabelle mit fünf Spalten wie "Load"
    rows,
    ["filter_id", "metric", "event_time_ms", "value_mwh", "source_file"],
)
print("wind rows", wind_df.count())
wind_df.write.mode("append").saveAsTable("bronze.smard_raw") # mode=append screibt

spark.table("bronze.smard_raw").groupBy("metric").count().show() # überprüfen (a check)

In [0]:
spark.table("bronze.smard_raw").groupBy("metric").count().show()
spark.table("gold.daily_load").selectExpr("min(day)", "max(day)", "count(*)").show()

## Silver neu, aus der ganzen Bronze-Tabelle, "Load" und "Wind".

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("bronze.smard_raw") # lesen bronze.smard_raw

silver = (
    bronze.withColumn(
        "event_time",
        F.from_utc_timestamp(F.to_timestamp(F.col("event_time_ms") / 1000), "Europe/Berlin"), # passt Zeitzone an
    )
    .withColumn("is_missing_value", F.col("value_mwh").isNull()) # markiere NULL
    .dropDuplicates(["metric", "event_time"]) # entferne Duplikate
)

silver.groupBy("metric").count().show() # muss beide metrics zeigen; min/max sollen Juni bis Ende Aug sein.
print("min", silver.agg(F.min("event_time")).first()[0]) 
print("max", silver.agg(F.max("event_time")).first()[0])


silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.electricity_hourly")

## Gold daily_energy_mix neu, nur Stunden mit "Load"

To Do -
1. "silver.electricity_hourly" hat zwei Zeilen für dieselbe(the same) Stunde: eine "Load", eine "Wind". Pivot "silver.electricity_hourly" macht eine Zeile pro Stunde: eine "Load" und eine "Wind" weil ich kann "Wind" nicht durch(by) "Load" teilen(divide), solange(as long as) die Zahlen nicht in derselben(of the same) Zeile stehen.
2. Stunde 00:00 wind_share = 9206 / 37035 = 0.25
3. 
 

In [0]:
from pyspark.sql import functions as F

hourly = ( # 
    spark.table("silver.electricity_hourly")
    .groupBy("event_time")
    .pivot("metric", ["load", "wind_onshore"]) # macht aus zwei metrics zwei Spalten in deselben Stunde
    .agg(F.first("value_mwh"))
    .filter(F.col("load").isNotNull()) # Stunden ohne "Load" werden entfernt.
)

gold = (
    hourly.withColumn("wind_share", F.col("wind_onshore") / F.col("load")) # wind_share = wind / total_load
    .groupBy(F.to_date("event_time").alias("day"))
    .agg(
        F.count("*").alias("hours"),
        F.avg("load").alias("avg_load_mwh"),
        F.avg("wind_onshore").alias("avg_wind_mwh"),
        F.avg("wind_share").alias("avg_wind_share"),
        F.max("wind_share").alias("max_wind_share"),
    )
)

print("days", gold.count())
gold.orderBy("day").show(5)

# daily_load bleibt unangetastet
gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.daily_energy_mix")